In [1]:
!pip install cdsapi

In [12]:
import os

# Actualizar el archivo .cdsapirc al nuevo formato (SIN uid, SOLO url y key)
home = os.path.expanduser("~")
filepath = os.path.join(home, '.cdsapirc')

# Tu API key actual
nueva_api_key = "3991abaa-caa0-4cd5-95e1-8bebf184e74c"

# Nuevo formato: SOLO url y key (sin uid)
content = f"""url: https://cds.climate.copernicus.eu/api
key: {nueva_api_key}
"""

with open(filepath, 'w') as f:
    f.write(content)

print(f"✅ Archivo creado en: {filepath}")

✅ Archivo creado en: /Users/antonio/.cdsapirc


In [14]:
import os
import cdsapi
import xarray as xr
import pandas as pd

# Inicializar cliente de Copernicus
c = cdsapi.Client()

# ==============================================================================
# CONFIGURACIÓN DE PARÁMETROS
# ==============================================================================
YEARS = [str(y) for y in range(2018, 2025)]
MONTHS = [f"{m:02d}" for m in range(1, 13)]
DAYS = [f"{d:02d}" for d in range(1, 32)]
TIMES = [f"{h:02d}:00" for h in range(24)]

# Coordenadas
LAT_LAND, LON_LAND = 21.0, -86.9
LAT_OCEAN, LON_OCEAN = 21.0, -86.75

# Áreas ajustadas (formato [N, W, S, E])
AREA_LAND = [LAT_LAND + 0.1, LON_LAND - 0.1, LAT_LAND - 0.1, LON_LAND + 0.1]
AREA_OCEAN = [LAT_OCEAN + 0.3, LON_OCEAN - 0.3, LAT_OCEAN - 0.3, LON_OCEAN + 0.3]

os.makedirs("data_era5_timeseries", exist_ok=True)

# CORRECCIÓN: Variables correctas para ERA5-Land
VARS_GROUP_A = [
    '2m_temperature', '2m_dewpoint_temperature', 
    '10m_u_component_of_wind', '10m_v_component_of_wind',
    'soil_temperature_level_1', 'soil_temperature_level_2', 
    'soil_temperature_level_3', 'soil_temperature_level_4'
]

VARS_GROUP_B = ['total_precipitation', 'surface_solar_radiation_downwards']

# CORRECCIÓN: Variables correctas para ERA5 single-levels (océano)
VARS_GROUP_C = [
    'significant_height_of_combined_wind_waves_and_swell',
    'mean_wave_direction', 'mean_wave_period',
    'significant_height_of_total_swell'
]

# ==============================================================================
# FUNCIONES AUXILIARES
# ==============================================================================
def download_timeseries(dataset, variables, year, area, filename):
    """Descarga datos horarios para un año completo."""
    if os.path.exists(filename):
        print(f"  -> El archivo {filename} ya existe. Omitiendo descarga.")
        return True

    print(f"  -> Descargando año {year} para {filename}...")

    try:
        # Parámetros base
        request_params = {
            'variable': variables,
            'year': year,
            'month': MONTHS,
            'day': DAYS,
            'time': TIMES,
            'format': 'netcdf'  # CORRECCIÓN: 'format' en lugar de 'data_format'
        }

        # CORRECCIÓN: Añadir product_type según el dataset
        if dataset == 'reanalysis-era5-single-levels':
            request_params['product_type'] = 'reanalysis'
            
            # CORRECCIÓN: Para ERA5 single-levels, necesitamos especificar el área
            if area:
                request_params['area'] = area
        else:
            # Para ERA5-Land, el área se especifica de manera diferente
            if area:
                request_params['area'] = area

        # Realizar la descarga
        c.retrieve(dataset, request_params, filename)
        print(f"  -> Descarga completada: {filename}")
        return True
        
    except Exception as e:
        print(f"  -> ERROR descargando {filename}: {str(e)}")
        return False

def clean_dataframe(df):
    """Limpia columnas innecesarias del formato timeseries."""
    cols_to_drop = ['latitude', 'longitude', 'number', 'step', 'surface', 
                    'valid_time', 'depth', 'expver']
    return df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# ==============================================================================
# DESCARGA Y PROCESAMIENTO
# ==============================================================================
print("\n========== INICIANDO DESCARDA OPTIMIZADA ==========")

all_data = []

for year in YEARS:
    print(f"\n--- Procesando Año {year} ---")
    
    # Nombres de archivo
    file_a = f"data_era5_timeseries/era5land_groupA_{year}.nc"
    file_b = f"data_era5_timeseries/era5land_groupB_{year}.nc"
    file_c = f"data_era5_timeseries/era5_single_levels_groupC_{year}.nc"

    # Descargar los grupos
    success_a = download_timeseries('reanalysis-era5-land', VARS_GROUP_A, year, AREA_LAND, file_a)
    success_b = download_timeseries('reanalysis-era5-land', VARS_GROUP_B, year, AREA_LAND, file_b)
    success_c = download_timeseries('reanalysis-era5-single-levels', VARS_GROUP_C, year, AREA_OCEAN, file_c)

    # Verificar que todas las descargas fueron exitosas
    if not (success_a and success_b and success_c):
        print(f"  -> Advertencia: Algunas descargas fallaron para el año {year}")
        continue

    # Procesar los datos
    try:
        print("  -> Abriendo y procesando los archivos NetCDF...")
        
        # Abrir y seleccionar puntos
        ds_a = xr.open_dataset(file_a).sel(latitude=LAT_LAND, longitude=LON_LAND, method='nearest')
        ds_b = xr.open_dataset(file_b).sel(latitude=LAT_LAND, longitude=LON_LAND, method='nearest')
        ds_c = xr.open_dataset(file_c).sel(latitude=LAT_OCEAN, longitude=LON_OCEAN, method='nearest')

        # Convertir a DataFrame
        df_a = clean_dataframe(ds_a.to_dataframe().reset_index())
        df_b = clean_dataframe(ds_b.to_dataframe().reset_index())
        df_c = clean_dataframe(ds_c.to_dataframe().reset_index())

        # CORRECCIÓN: Conversiones con nombres correctos de variables
        if 'tp' in df_b.columns:
            df_b['precipitacion_mm'] = df_b['tp'] * 1000.0
            df_b = df_b.drop(columns=['tp'])

        if 'ssrd' in df_b.columns:
            # Convertir de J/m² a W/m² (dividir por segundos en la hora)
            df_b['radiacion_solar_wm2'] = df_b['ssrd'] / 3600.0
            df_b = df_b.drop(columns=['ssrd'])

        # CORRECCIÓN: Convertir temperatura de Kelvin a Celsius si es necesario
        if 't2m' in df_a.columns:
            df_a['temperatura_2m_c'] = df_a['t2m'] - 273.15
            
        if 'd2m' in df_a.columns:
            df_a['punto_rocio_2m_c'] = df_a['d2m'] - 273.15

        # Unir todos los datos del año
        df_merged = pd.merge(df_a, df_b, on='time', how='inner')
        df_merged = pd.merge(df_merged, df_c, on='time', how='inner')

        all_data.append(df_merged)

        # Cerrar archivos
        ds_a.close()
        ds_b.close()
        ds_c.close()
        
        print(f"  -> Año {year} procesado correctamente")
        
    except Exception as e:
        print(f"  -> Error procesando año {year}: {str(e)}")
        continue

# Unir todos los años
if all_data:
    print("\n  -> Unificando datos de todos los años...")
    df_final = pd.concat(all_data, ignore_index=True)
    
    # Ordenar por tiempo y eliminar duplicados
    df_final = df_final.drop_duplicates(subset=['time']).sort_values('time').reset_index(drop=True)
    
    # Guardar a CSV
    output_csv = "serie_tiempo_campamento_tamul.csv"
    df_final.to_csv(output_csv, index=False)
    
    print(f"\n¡Proceso finalizado con éxito!")
    print(f"Total de registros: {len(df_final)}")
    print(f"Período: {df_final['time'].min()} a {df_final['time'].max()}")
    print(f"Archivo guardado: {output_csv}")
    
    # Mostrar primeras filas
    print("\nPrimeros registros:")
    print(df_final.head())
    
else:
    print("\nNo se pudieron procesar datos para ningún año.")


========== INICIANDO DESCARDA OPTIMIZADA ==========

--- Procesando Año 2018 ---
  -> Descargando año 2018 para data_era5_timeseries/era5land_groupA_2018.nc...
  -> ERROR descargando data_era5_timeseries/era5land_groupA_2018.nc: 403 Client Error: Forbidden for url: https://cds.climate.copernicus.eu/api/retrieve/v1/processes/reanalysis-era5-land/execution
cost limits exceeded
Your request is too large, please reduce your selection.
  -> Descargando año 2018 para data_era5_timeseries/era5land_groupB_2018.nc...
  -> ERROR descargando data_era5_timeseries/era5land_groupB_2018.nc: 403 Client Error: Forbidden for url: https://cds.climate.copernicus.eu/api/retrieve/v1/processes/reanalysis-era5-land/execution
cost limits exceeded
Your request is too large, please reduce your selection.
  -> Descargando año 2018 para data_era5_timeseries/era5_single_levels_groupC_2018.nc...
  -> ERROR descargando data_era5_timeseries/era5_single_levels_groupC_2018.nc: 403 Client Error: Forbidden for url: https

In [14]:
import os
import cdsapi
import xarray as xr
import pandas as pd

# Inicializar cliente de Copernicus
c = cdsapi.Client()

# ==============================================================================
# CONFIGURACIÓN DE PARÁMETROS - SOLO UN DÍA ESPECÍFICO
# ==============================================================================
# Configurar para un día específico: 20 de junio de 2024
YEAR = '2024'
MONTH = '06'
DAY = '20'
TIMES = [f"{h:02d}:00" for h in range(24)]  # Todas las horas del día

# Coordenadas
LAT_LAND, LON_LAND = 21.0, -86.9
LAT_OCEAN, LON_OCEAN = 21.0, -86.75

# Áreas ajustadas (formato [N, W, S, E])
AREA_LAND = [LAT_LAND + 0.1, LON_LAND - 0.1, LAT_LAND - 0.1, LON_LAND + 0.1]
AREA_OCEAN = [LAT_OCEAN + 0.3, LON_OCEAN - 0.3, LAT_OCEAN - 0.3, LON_OCEAN + 0.3]

os.makedirs("data_era5_timeseries", exist_ok=True)

# Variables para ERA5-Land
VARS_GROUP_A = [
    '2m_temperature', '2m_dewpoint_temperature', 
    '10m_u_component_of_wind', '10m_v_component_of_wind',
    'soil_temperature_level_1', 'soil_temperature_level_2', 
    'soil_temperature_level_3'
]

VARS_GROUP_B = ['total_precipitation', 'surface_solar_radiation_downwards']

# Variables para ERA5 single-levels (océano)
VARS_GROUP_C = [
    'significant_height_of_combined_wind_waves_and_swell',
    'mean_wave_direction', 'mean_wave_period',
    'significant_height_of_total_swell'
]

# ==============================================================================
# FUNCIONES AUXILIARES
# ==============================================================================
def download_day(dataset, variables, year, month, day, area, filename):
    """Descarga datos para un día específico."""
    if os.path.exists(filename):
        print(f"  -> El archivo {filename} ya existe. Omitiendo descarga.")
        return True

    print(f"  -> Descargando {year}-{month}-{day} para {filename}...")

    try:
        request_params = {
            'variable': variables,
            'year': year,
            'month': month,
            'day': day,
            'time': TIMES,
            'format': 'netcdf'
        }

        if dataset == 'reanalysis-era5-single-levels':
            request_params['product_type'] = 'reanalysis'
            
        if area:
            request_params['area'] = area

        c.retrieve(dataset, request_params, filename)
        print(f"  -> Descarga completada: {filename}")
        return True
        
    except Exception as e:
        print(f"  -> ERROR descargando {filename}: {str(e)}")
        return False

def clean_dataframe(df):
    """Limpia columnas innecesarias del formato timeseries."""
    cols_to_drop = ['latitude', 'longitude', 'number', 'step', 'surface', 
                    'valid_time', 'depth', 'expver']
    return df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# ==============================================================================
# DESCARGA Y PROCESAMIENTO - SOLO UN DÍA
# ==============================================================================
print("\n========== DESCARGANDO DATOS PARA 20 DE JUNIO DE 2024 ==========")
print(f"Fecha: {YEAR}-{MONTH}-{DAY}")
print(f"Coordenadas Tierra: {LAT_LAND}, {LON_LAND}")
print(f"Coordenadas Océano: {LAT_OCEAN}, {LON_OCEAN}")

# Nombres de archivo para este día específico
file_a = f"data_era5_timeseries/era5land_groupA_{YEAR}{MONTH}{DAY}.nc"
file_b = f"data_era5_timeseries/era5land_groupB_{YEAR}{MONTH}{DAY}.nc"
file_c = f"data_era5_timeseries/era5_single_levels_groupC_{YEAR}{MONTH}{DAY}.nc"

# Descargar los grupos para este día
print("\n--- Iniciando descargas ---")
success_a = download_day('reanalysis-era5-land', VARS_GROUP_A, YEAR, MONTH, DAY, AREA_LAND, file_a)
success_b = download_day('reanalysis-era5-land', VARS_GROUP_B, YEAR, MONTH, DAY, AREA_LAND, file_b)
success_c = download_day('reanalysis-era5-single-levels', VARS_GROUP_C, YEAR, MONTH, DAY, AREA_OCEAN, file_c)

# Verificar que todas las descargas fueron exitosas
if not (success_a and success_b and success_c):
    print("\n❌ Error: Algunas descargas fallaron")
else:
    # Procesar los datos
    try:
        print("\n--- Procesando datos ---")
        
        # Abrir y seleccionar puntos
        print("Abriendo archivos NetCDF...")
        ds_a = xr.open_dataset(file_a).sel(latitude=LAT_LAND, longitude=LON_LAND, method='nearest')
        ds_b = xr.open_dataset(file_b).sel(latitude=LAT_LAND, longitude=LON_LAND, method='nearest')
        ds_c = xr.open_dataset(file_c).sel(latitude=LAT_OCEAN, longitude=LON_OCEAN, method='nearest')

        # Convertir a DataFrame
        print("Convirtiendo a DataFrames...")
        df_a = clean_dataframe(ds_a.to_dataframe().reset_index())
        df_b = clean_dataframe(ds_b.to_dataframe().reset_index())
        df_c = clean_dataframe(ds_c.to_dataframe().reset_index())

        # Conversiones de unidades
        print("Aplicando conversiones de unidades...")
        
        # Precipitación: de metros a mm
        if 'tp' in df_b.columns:
            df_b['precipitacion_mm'] = df_b['tp'] * 1000.0
            df_b = df_b.drop(columns=['tp'])
            print("  - Precipitación convertida a mm")

        # Radiación solar: de J/m² a W/m² (promedio horario)
        if 'ssrd' in df_b.columns:
            df_b['radiacion_solar_wm2'] = df_b['ssrd'] / 3600.0
            df_b = df_b.drop(columns=['ssrd'])
            print("  - Radiación solar convertida a W/m²")

        # Temperatura: de Kelvin a Celsius
        if 't2m' in df_a.columns:
            df_a['temperatura_2m_c'] = df_a['t2m'] - 273.15
            print("  - Temperatura convertida a °C")
            
        if 'd2m' in df_a.columns:
            df_a['punto_rocio_2m_c'] = df_a['d2m'] - 273.15

        # Unir todos los datos
        print("Uniendo dataframes...")
        df_final = pd.merge(df_a, df_b, on='time', how='inner')
        df_final = pd.merge(df_final, df_c, on='time', how='inner')

        # Ordenar por tiempo
        df_final = df_final.sort_values('time').reset_index(drop=True)

        # Cerrar archivos
        ds_a.close()
        ds_b.close()
        ds_c.close()
        
        # Guardar a CSV
        output_csv = f"datos_campamento_tamul_{YEAR}{MONTH}{DAY}.csv"
        df_final.to_csv(output_csv, index=False)
        
        print(f"\n✅ ¡Proceso completado exitosamente!")
        print(f"📊 Total de registros: {len(df_final)}")
        print(f"📁 Archivo guardado: {output_csv}")
        
        # Mostrar resumen de los datos
        print("\n📈 Resumen de datos:")
        print(df_final[['time', 'temperatura_2m_c', 'precipitacion_mm', 'radiacion_solar_wm2']].head())
        
        print("\n📊 Estadísticas básicas:")
        print(df_final[['temperatura_2m_c', 'precipitacion_mm', 'radiacion_solar_wm2']].describe())
        
    except Exception as e:
        print(f"\n❌ Error procesando datos: {str(e)}")


========== DESCARGANDO DATOS PARA 20 DE JUNIO DE 2024 ==========
Fecha: 2024-06-20
Coordenadas Tierra: 21.0, -86.9
Coordenadas Océano: 21.0, -86.75

--- Iniciando descargas ---
  -> Descargando 2024-06-20 para data_era5_timeseries/era5land_groupA_20240620.nc...


2026-03-04 17:54:06,979 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land-timeseries?tab=overview)
2026-03-04 17:54:06,981 INFO Request ID is 81cb7a4b-4e9d-4a38-9f64-f3fe55b48ed7
2026-03-04 17:54:07,156 INFO status has been updated to accepted
2026-03-04 18:00:28,891 INFO status has been updated to successful


14ee249de3d4572f8bb68c33edae42c7.zip:   0%|          | 0.00/80.4k [00:00<?, ?B/s]

  -> Descarga completada: data_era5_timeseries/era5land_groupA_20240620.nc
  -> Descargando 2024-06-20 para data_era5_timeseries/era5land_groupB_20240620.nc...


2026-03-04 18:00:31,545 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land-timeseries?tab=overview)
2026-03-04 18:00:31,546 INFO Request ID is 272d6996-7f30-4476-912b-edce94e26de0
2026-03-04 18:00:31,881 INFO status has been updated to accepted
2026-03-04 18:00:46,283 INFO status has been updated to running
2026-03-04 18:00:54,034 INFO status has been updated to successful


56fccb80648646ec2e1e095ae9bc1d92.zip:   0%|          | 0.00/33.9k [00:00<?, ?B/s]

  -> Descarga completada: data_era5_timeseries/era5land_groupB_20240620.nc
  -> Descargando 2024-06-20 para data_era5_timeseries/era5_single_levels_groupC_20240620.nc...


2026-03-04 18:00:56,213 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-03-04 18:00:56,213 INFO Request ID is e4604a6f-f28a-4c1d-9e98-2a6aa17735e4
2026-03-04 18:00:56,366 INFO status has been updated to accepted
2026-03-04 18:01:05,186 INFO status has been updated to running
2026-03-04 18:01:18,188 INFO status has been updated to successful


f2c57333b36b6fcadcbddb5e0862a214.nc:   0%|          | 0.00/52.8k [00:00<?, ?B/s]

  -> Descarga completada: data_era5_timeseries/era5_single_levels_groupC_20240620.nc

--- Procesando datos ---
Abriendo archivos NetCDF...

❌ Error procesando datos: did not find a match in any of xarray's currently installed IO backends ['netcdf4']. Consider explicitly selecting one of the installed engines via the ``engine`` parameter, or installing additional IO dependencies, see:
https://docs.xarray.dev/en/stable/getting-started-guide/installing.html
https://docs.xarray.dev/en/stable/user-guide/io.html


In [1]:
import os
import xarray as xr
import pandas as pd
import glob
from datetime import datetime

# ==============================================================================
# CONFIGURACIÓN
# ==============================================================================
# Coordenadas de los puntos de interés
LAT_LAND, LON_LAND = 21.0, -86.9      # Punto en tierra (Campamento Tamul)
LAT_OCEAN, LON_OCEAN = 21.0, -86.75   # Punto en océano

# Directorios
INPUT_DIR = "data_era5_timeseries"     # Directorio con los archivos .nc
OUTPUT_DIR = "datos_procesados"        # Directorio para los CSV

# Crear directorio de salida si no existe
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==============================================================================
# FUNCIONES DE TRANSFORMACIÓN
# ==============================================================================

def convertir_temperatura_kelvin_a_celsius(df, col_temp='t2m', col_rocio='d2m'):
    """Convierte temperatura de Kelvin a Celsius."""
    if col_temp in df.columns:
        df['temperatura_2m_c'] = df[col_temp] - 273.15
        df = df.drop(columns=[col_temp])
        print("  ✓ Temperatura convertida a °C")
    
    if col_rocio in df.columns:
        df['punto_rocio_2m_c'] = df[col_rocio] - 273.15
        df = df.drop(columns=[col_rocio])
        print("  ✓ Punto de rocío convertido a °C")
    
    return df

def convertir_precipitacion(df, col_precip='tp'):
    """Convierte precipitación de metros a milímetros."""
    if col_precip in df.columns:
        df['precipitacion_mm'] = df[col_precip] * 1000.0
        df = df.drop(columns=[col_precip])
        print("  ✓ Precipitación convertida a mm")
    return df

def convertir_radiacion_solar(df, col_rad='ssrd'):
    """Convierte radiación solar de J/m² a W/m² (promedio horario)."""
    if col_rad in df.columns:
        df['radiacion_solar_wm2'] = df[col_rad] / 3600.0
        df = df.drop(columns=[col_rad])
        print("  ✓ Radiación solar convertida a W/m²")
    return df

def convertir_viento(df, col_u='u10', col_v='v10'):
    """Calcula velocidad y dirección del viento a partir de componentes U y V."""
    if col_u in df.columns and col_v in df.columns:
        # Velocidad del viento (m/s)
        df['viento_velocidad_ms'] = (df[col_u]**2 + df[col_v]**2)**0.5
        
        # Dirección del viento (grados, de donde viene)
        df['viento_direccion_grados'] = (270 - (180/3.14159) * 
                                          np.arctan2(df[col_v], df[col_u])) % 360
        
        df = df.drop(columns=[col_u, col_v])
        print("  ✓ Velocidad y dirección del viento calculadas")
    return df

def convertir_oleaje(df):
    """Procesa variables de oleaje."""
    if 'swh' in df.columns:
        df = df.rename(columns={'swh': 'altura_oleaje_m'})
        print("  ✓ Altura de oleaje renombrada")
    
    if 'mwd' in df.columns:
        df = df.rename(columns={'mwd': 'direccion_oleaje_grados'})
        print("  ✓ Dirección de oleaje renombrada")
    
    if 'mwp' in df.columns:
        df = df.rename(columns={'mwp': 'periodo_oleaje_s'})
        print("  ✓ Período de oleaje renombrada")
    
    return df

def clean_dataframe(df):
    """Limpia columnas innecesarias."""
    cols_to_drop = ['latitude', 'longitude', 'number', 'step', 'surface', 
                    'valid_time', 'depth', 'expver']
    existing_cols = [c for c in cols_to_drop if c in df.columns]
    
    if existing_cols:
        df = df.drop(columns=existing_cols)
        print(f"  ✓ Columnas eliminadas: {existing_cols}")
    
    return df

def procesar_archivo(nc_file, punto, es_oceano=False):
    """
    Procesa un archivo NetCDF y devuelve un DataFrame.
    
    Args:
        nc_file: Ruta al archivo .nc
        punto: Tupla (lat, lon) del punto a extraer
        es_oceano: Si es True, usa nombres de variables oceánicas
    """
    print(f"\n📄 Procesando: {os.path.basename(nc_file)}")
    
    try:
        # Abrir archivo
        ds = xr.open_dataset(nc_file, engine='netcdf4')
        
        # Extraer punto más cercano
        lat, lon = punto
        ds_punto = ds.sel(latitude=lat, longitude=lon, method='nearest')
        
        # Convertir a DataFrame
        df = ds_punto.to_dataframe().reset_index()
        
        # Cerrar dataset
        ds.close()
        
        # Limpiar columnas innecesarias
        df = clean_dataframe(df)
        
        # Aplicar conversiones según el tipo de archivo
        if 'soil_temperature' in nc_file or 'groupA' in nc_file:
            # Archivo de temperatura y suelo
            df = convertir_temperatura_kelvin_a_celsius(df)
            df = convertir_viento(df)
            
        elif 'groupB' in nc_file:
            # Archivo de precipitación y radiación
            df = convertir_precipitacion(df)
            df = convertir_radiacion_solar(df)
            
        elif 'groupC' in nc_file or 'single_levels' in nc_file:
            # Archivo de oleaje
            df = convertir_oleaje(df)
        
        print(f"  ✅ Procesado: {len(df)} registros")
        return df
        
    except Exception as e:
        print(f"  ❌ Error: {str(e)}")
        return None

def combinar_archivos_por_fecha():
    """
    Busca archivos por fecha y combina los grupos A, B y C para cada fecha.
    """
    # Encontrar todas las fechas únicas
    archivos = glob.glob(os.path.join(INPUT_DIR, "*.nc"))
    
    # Extraer fechas de los nombres de archivo
    fechas = set()
    for archivo in archivos:
        nombre = os.path.basename(archivo)
        # Buscar patrón YYYYMMDD en el nombre
        import re
        match = re.search(r'(\d{8})', nombre)
        if match:
            fechas.add(match.group(1))
    
    print(f"🔍 Encontradas {len(fechas)} fechas para procesar")
    print(f"📅 Fechas: {sorted(fechas)}")
    
    todos_datos = []
    
    for fecha in sorted(fechas):
        print(f"\n{'='*50}")
        print(f"📅 Procesando fecha: {fecha}")
        print(f"{'='*50}")
        
        # Buscar archivos para esta fecha
        archivo_a = glob.glob(os.path.join(INPUT_DIR, f"*groupA*{fecha}*.nc"))
        archivo_b = glob.glob(os.path.join(INPUT_DIR, f"*groupB*{fecha}*.nc"))
        archivo_c = glob.glob(os.path.join(INPUT_DIR, f"*groupC*{fecha}*.nc"))
        
        dfs_fecha = []
        
        # Procesar grupo A (tierra)
        if archivo_a:
            df_a = procesar_archivo(archivo_a[0], (LAT_LAND, LON_LAND))
            if df_a is not None:
                dfs_fecha.append(df_a)
        
        # Procesar grupo B (tierra)
        if archivo_b:
            df_b = procesar_archivo(archivo_b[0], (LAT_LAND, LON_LAND))
            if df_b is not None:
                dfs_fecha.append(df_b)
        
        # Procesar grupo C (océano)
        if archivo_c:
            df_c = procesar_archivo(archivo_c[0], (LAT_OCEAN, LON_OCEAN), es_oceano=True)
            if df_c is not None:
                dfs_fecha.append(df_c)
        
        # Combinar todos los DataFrames de la fecha
        if len(dfs_fecha) >= 3:
            print("\n🔄 Combinando grupos para esta fecha...")
            
            # Merge secuencial
            df_combinado = dfs_fecha[0]
            for df_extra in dfs_fecha[1:]:
                df_combinado = pd.merge(df_combinado, df_extra, on='time', how='outer')
            
            # Ordenar por tiempo
            df_combinado = df_combinado.sort_values('time').reset_index(drop=True)
            
            # Guardar archivo individual por fecha
            output_file = os.path.join(OUTPUT_DIR, f"datos_tamul_{fecha}.csv")
            df_combinado.to_csv(output_file, index=False)
            print(f"  💾 Guardado: {output_file}")
            
            todos_datos.append(df_combinado)
        else:
            print(f"  ⚠️ No se encontraron todos los grupos para {fecha}")
    
    return todos_datos

# ==============================================================================
# EJECUCIÓN PRINCIPAL
# ==============================================================================
if __name__ == "__main__":
    print("="*60)
    print("🚀 INICIANDO TRANSFORMACIÓN DE ARCHIVOS ERA5")
    print("="*60)
    print(f"📂 Directorio de entrada: {INPUT_DIR}")
    print(f"📂 Directorio de salida: {OUTPUT_DIR}")
    print(f"📍 Punto tierra: {LAT_LAND}, {LON_LAND}")
    print(f"📍 Punto océano: {LAT_OCEAN}, {LON_OCEAN}")
    
    # Procesar todos los archivos
    todos_datos = combinar_archivos_por_fecha()
    
    # Combinar todos en un solo archivo si hay múltiples fechas
    if len(todos_datos) > 1:
        print(f"\n{'='*50}")
        print("🔄 COMBINANDO TODAS LAS FECHAS")
        print(f"{'='*50}")
        
        df_final = pd.concat(todos_datos, ignore_index=True)
        df_final = df_final.drop_duplicates(subset=['time']).sort_values('time').reset_index(drop=True)
        
        # Guardar archivo completo
        output_completo = os.path.join(OUTPUT_DIR, "datos_tamul_completo.csv")
        df_final.to_csv(output_completo, index=False)
        
        print(f"\n✅ Archivo completo guardado: {output_completo}")
        print(f"📊 Total de registros: {len(df_final)}")
        print(f"📅 Período: {df_final['time'].min()} a {df_final['time'].max()}")
        
        # Mostrar estadísticas básicas
        print(f"\n📈 Estadísticas básicas:")
        columnas_numericas = df_final.select_dtypes(include=[np.number]).columns
        print(df_final[columnas_numericas].describe())
        
    elif len(todos_datos) == 1:
        print(f"\n✅ Procesamiento completado. Datos guardados en {OUTPUT_DIR}")
    else:
        print(f"\n❌ No se procesó ningún archivo")
    
    print(f"\n{'='*60}")
    print("🏁 PROCESO FINALIZADO")
    print(f"{'='*60}")

🚀 INICIANDO TRANSFORMACIÓN DE ARCHIVOS ERA5
📂 Directorio de entrada: data_era5_timeseries
📂 Directorio de salida: datos_procesados
📍 Punto tierra: 21.0, -86.9
📍 Punto océano: 21.0, -86.75
🔍 Encontradas 1 fechas para procesar
📅 Fechas: ['20240620']

📅 Procesando fecha: 20240620

📄 Procesando: era5land_groupA_20240620.nc
  ❌ Error: [Errno -51] NetCDF: Unknown file format: '/Users/antonio/Downloads/data_era5_timeseries/era5land_groupA_20240620.nc'

📄 Procesando: era5land_groupB_20240620.nc
  ❌ Error: [Errno -51] NetCDF: Unknown file format: '/Users/antonio/Downloads/data_era5_timeseries/era5land_groupB_20240620.nc'

📄 Procesando: era5_single_levels_groupC_20240620.nc
  ✓ Columnas eliminadas: ['latitude', 'longitude', 'number', 'valid_time', 'expver']
  ✓ Altura de oleaje renombrada
  ✓ Dirección de oleaje renombrada
  ✓ Período de oleaje renombrada
  ✅ Procesado: 24 registros
  ⚠️ No se encontraron todos los grupos para 20240620

❌ No se procesó ningún archivo

🏁 PROCESO FINALIZADO


In [2]:
import os
import glob

# Ruta a tus archivos
download_dir = "/Users/antonio/Downloads/data_era5_timeseries"

# Listar todos los archivos
archivos = glob.glob(os.path.join(download_dir, "*.nc"))
print(f"Archivos encontrados: {len(archivos)}")

for archivo in archivos:
    tamaño = os.path.getsize(archivo)
    print(f"\n📁 {os.path.basename(archivo)}")
    print(f"   Tamaño: {tamaño} bytes ({tamaño/1024:.2f} KB)")
    
    # Verificar los primeros bytes del archivo (magic number)
    try:
        with open(archivo, 'rb') as f:
            header = f.read(8)
            print(f"   Header (hex): {header.hex()}")
            print(f"   Header (ascii): {header}")
            
            # NetCDF3 comienza con 'CDF' (0x43444601)
            # NetCDF4/HDF5 comienza con \x89HDF (0x894844460d0a1a0a)
            if header.startswith(b'CDF'):
                print("   ✅ Formato: NetCDF3")
            elif header.startswith(b'\x89HDF'):
                print("   ✅ Formato: NetCDF4/HDF5")
            elif header.startswith(b'<!DOCTYPE') or header.startswith(b'<html'):
                print("   ❌ Es un archivo HTML (error de API)")
            elif len(header) == 0:
                print("   ❌ Archivo vacío")
            else:
                print("   ❌ Formato desconocido")
    except Exception as e:
        print(f"   ❌ Error leyendo: {e}")

Archivos encontrados: 3

📁 era5_single_levels_groupC_20240620.nc
   Tamaño: 54076 bytes (52.81 KB)
   Header (hex): 894844460d0a1a0a
   Header (ascii): b'\x89HDF\r\n\x1a\n'
   ✅ Formato: NetCDF4/HDF5

📁 era5land_groupB_20240620.nc
   Tamaño: 34752 bytes (33.94 KB)
   Header (hex): 504b030414000000
   Header (ascii): b'PK\x03\x04\x14\x00\x00\x00'
   ❌ Formato desconocido

📁 era5land_groupA_20240620.nc
   Tamaño: 82359 bytes (80.43 KB)
   Header (hex): 504b030414000000
   Header (ascii): b'PK\x03\x04\x14\x00\x00\x00'
   ❌ Formato desconocido


In [13]:
import xarray as xr
import pandas as pd
import numpy as np
import os

# Configuración
download_dir = "/Users/antonio/Downloads"
output_dir = "/Users/antonio/Downloads/datos_procesados"
os.makedirs(output_dir, exist_ok=True)

# Coordenadas del punto oceánico
LAT_OCEAN, LON_OCEAN = 21.0, -86.75

# Archivo a procesar
archivo_c = os.path.join(download_dir, "data_0.nc")

print("="*60)
print("🌊 PROCESANDO DATOS DE OLEAJE (GRUPO C)")
print("="*60)

try:
    # 1. Abrir el archivo NetCDF
    print(f"\n📂 Abriendo archivo: {os.path.basename(archivo_c)}")
    ds = xr.open_dataset(archivo_c, engine='netcdf4')
    
    # 2. Mostrar información del dataset
    print(f"\n📊 Información del dataset:")
    print(f"   Dimensiones: {ds.sizes}")  # Cambiado de dims a sizes
    print(f"   Variables: {list(ds.data_vars.keys())}")
    print(f"   Coordenadas: {list(ds.coords.keys())}")
    
    # 3. Extraer el punto más cercano
    print(f"\n📍 Extrayendo punto: lat={LAT_OCEAN}, lon={LON_OCEAN}")
    ds_punto = ds.sel(latitude=LAT_OCEAN, longitude=LON_OCEAN, method='nearest')
    
    # 4. Convertir a DataFrame y resetear índice
    print(f"\n🔄 Convirtiendo a DataFrame...")
    df = ds_punto.to_dataframe().reset_index()
    
    # 5. MOSTRAR COLUMNAS DISPONIBLES
    print(f"\n📋 COLUMNAS DISPONIBLES EN EL DATAFRAME:")
    print(f"   {list(df.columns)}")
    
    # 6. Limpiar columnas innecesarias
    cols_to_drop = ['latitude', 'longitude', 'number', 'valid_time', 'expver']
    columnas_existentes = [col for col in cols_to_drop if col in df.columns]
    
    if columnas_existentes:
        df = df.drop(columns=columnas_existentes)
        print(f"\n   Eliminadas columnas: {columnas_existentes}")
    
    # 7. Volver a mostrar columnas después de limpieza
    print(f"\n📋 COLUMNAS DESPUÉS DE LIMPIEZA:")
    print(f"   {list(df.columns)}")
    
    # 8. Renombrar variables
    rename_map = {
        'swh': 'altura_oleaje_m',
        'mwd': 'direccion_oleaje_grados',
        'mwp': 'periodo_oleaje_s',
    }
    
    columnas_a_renombrar = {k: v for k, v in rename_map.items() if k in df.columns}
    if columnas_a_renombrar:
        df = df.rename(columns=columnas_a_renombrar)
        print(f"\n   Renombradas variables: {list(columnas_a_renombrar.values())}")
    
    # 9. Verificar que la columna 'time' existe
    print(f"\n⏰ Verificando columna de tiempo:")
    if 'time' in df.columns:
        print(f"   ✅ Columna 'time' encontrada")
        print(f"   Tipo de dato: {df['time'].dtype}")
        print(f"   Rango: {df['time'].min()} a {df['time'].max()}")
    else:
        # Buscar columna de tiempo con otro nombre
        posibles_tiempo = [col for col in df.columns if 'time' in col.lower()]
        if posibles_tiempo:
            print(f"   ⚠️ No se encontró 'time', pero sí: {posibles_tiempo}")
            # Renombrar la primera columna de tiempo encontrada
            df = df.rename(columns={posibles_tiempo[0]: 'time'})
            print(f"   Renombrada '{posibles_tiempo[0]}' a 'time'")
        else:
            print(f"   ❌ No se encontró ninguna columna de tiempo")
            print(f"   Columnas disponibles: {list(df.columns)}")
    
    # 10. Mostrar información del DataFrame
    print(f"\n📊 Información del DataFrame:")
    print(f"   Shape: {df.shape}")
    print(f"   Tipos de datos:")
    for col in df.columns:
        print(f"     - {col}: {df[col].dtype}")
    
    # 11. Guardar a CSV
    output_csv = os.path.join(output_dir, "datos_corruptos.csv")
    df.to_csv(output_csv, index=False)
    print(f"\n💾 Archivo guardado: {output_csv}")
    
    # 12. Mostrar las primeras filas (de manera segura)
    print(f"\n👀 Primeros 5 registros:")
    
    # Seleccionar columnas para mostrar
    columnas_mostrar = ['time'] if 'time' in df.columns else []
    columnas_mostrar += [col for col in df.columns if col != 'time' and col != 'index']
    
    if columnas_mostrar:
        print(df[columnas_mostrar].head())
    else:
        print(df.head())
    
    # 13. Estadísticas básicas
    print(f"\n📊 Estadísticas básicas:")
    variables_numericas = df.select_dtypes(include=[np.number]).columns
    for var in variables_numericas:
        if var != 'time':
            print(f"\n   {var}:")
            print(f"      Mínimo: {df[var].min():.3f}")
            print(f"      Máximo: {df[var].max():.3f}")
            print(f"      Promedio: {df[var].mean():.3f}")
    
    # 14. Cerrar el dataset
    ds.close()
    
    print(f"\n{'='*60}")
    print("✅ PROCESO COMPLETADO EXITOSAMENTE")
    print(f"{'='*60}")
    
except Exception as e:
    print(f"\n❌ Error: {str(e)}")
    import traceback
    traceback.print_exc()

🌊 PROCESANDO DATOS DE OLEAJE (GRUPO C)

📂 Abriendo archivo: data_0.nc

📊 Información del dataset:
   Dimensiones: Frozen({'valid_time': 24, 'latitude': 3, 'longitude': 3})
   Variables: ['tp', 'ssrd']
   Coordenadas: ['number', 'valid_time', 'latitude', 'longitude', 'expver']

📍 Extrayendo punto: lat=21.0, lon=-86.75

🔄 Convirtiendo a DataFrame...

📋 COLUMNAS DISPONIBLES EN EL DATAFRAME:
   ['valid_time', 'tp', 'ssrd', 'number', 'latitude', 'longitude', 'expver']

   Eliminadas columnas: ['latitude', 'longitude', 'number', 'valid_time', 'expver']

📋 COLUMNAS DESPUÉS DE LIMPIEZA:
   ['tp', 'ssrd']

⏰ Verificando columna de tiempo:
   ❌ No se encontró ninguna columna de tiempo
   Columnas disponibles: ['tp', 'ssrd']

📊 Información del DataFrame:
   Shape: (24, 2)
   Tipos de datos:
     - tp: float32
     - ssrd: float32

💾 Archivo guardado: /Users/antonio/Downloads/datos_procesados/datos_corruptos.csv

👀 Primeros 5 registros:
   tp  ssrd
0 NaN   NaN
1 NaN   NaN
2 NaN   NaN
3 NaN   NaN
4

In [2]:
import cdsapi
import os
import time
import glob
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ==============================================================================
# CONFIGURACIÓN
# ==============================================================================

# Fecha específica
YEAR = '2024'
MONTH = '09'
DAY = '20'
TIMES = [f"{h:02d}:00" for h in range(24)]

# Coordenadas
LAT_LAND, LON_LAND = 21.0, -86.9
LAT_OCEAN, LON_OCEAN = 21.0, -86.75

# Áreas (formato [N, W, S, E])
AREA_LAND = [LAT_LAND + 0.1, LON_LAND - 0.1, LAT_LAND - 0.1, LON_LAND + 0.1]
AREA_OCEAN = [LAT_OCEAN + 0.3, LON_OCEAN - 0.3, LAT_OCEAN - 0.3, LON_OCEAN + 0.3]

# Directorios
DOWNLOAD_DIR = "/Users/antonio/Downloads/data_era5_timeseries"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# Variables por grupo
VARS_GROUP_A = [
    '2m_temperature', 
    '2m_dewpoint_temperature', 
    '10m_u_component_of_wind',
    '10m_v_component_of_wind', 
    'soil_temperature_level_1',
    'soil_temperature_level_2', 
    'soil_temperature_level_3'
]

VARS_GROUP_B = [
    'total_precipitation', 
    'surface_solar_radiation_downwards'
]

VARS_GROUP_C = [
    'significant_height_of_combined_wind_waves_and_swell',
    'mean_wave_direction', 
    'mean_wave_period',
    'significant_height_of_total_swell'
]

# ==============================================================================
# FUNCIONES DE UTILIDAD
# ==============================================================================

def setup_retry_session():
    """Configura una sesión con reintentos para requests"""
    session = requests.Session()
    retries = Retry(
        total=5,
        backoff_factor=1,
        status_forcelist=[500, 502, 503, 504]
    )
    session.mount('https://', HTTPAdapter(max_retries=retries))
    return session

def verificar_archivo_nc(filepath):
    """Verifica si un archivo es un NetCDF válido leyendo su header"""
    if not os.path.exists(filepath):
        return False, "Archivo no existe"
    
    tamaño = os.path.getsize(filepath)
    if tamaño < 1000:  # Menos de 1KB probablemente está corrupto
        return False, f"Archivo demasiado pequeño ({tamaño} bytes)"
    
    try:
        with open(filepath, 'rb') as f:
            header = f.read(8)
        
        # NetCDF4/HDF5 comienza con \x89HDF
        if header.startswith(b'\x89HDF'):
            return True, f"NetCDF4 válido ({tamaño/1024:.2f} KB)"
        # NetCDF3 comienza con 'CDF'
        elif header.startswith(b'CDF'):
            return True, f"NetCDF3 válido ({tamaño/1024:.2f} KB)"
        # ZIP (error de API)
        elif header.startswith(b'PK'):
            return False, f"Archivo ZIP corrupto ({tamaño/1024:.2f} KB)"
        # HTML (error de API)
        elif header.startswith(b'<!') or header.startswith(b'<ht'):
            return False, f"Archivo HTML (error de API) ({tamaño/1024:.2f} KB)"
        else:
            return False, f"Formato desconocido: {header.hex()}"
            
    except Exception as e:
        return False, f"Error leyendo archivo: {str(e)}"

def descargar_con_verificacion(dataset, variables, year, month, day, area, filename, max_intentos=3):
    """
    Descarga un archivo con verificación y reintentos automáticos
    """
    print(f"\n{'='*60}")
    print(f"📥 Descargando: {os.path.basename(filename)}")
    print(f"{'='*60}")
    
    # Verificar si ya existe un archivo válido
    if os.path.exists(filename):
        es_valido, mensaje = verificar_archivo_nc(filename)
        if es_valido:
            print(f"   ✅ Archivo existente válido: {mensaje}")
            return True
        else:
            print(f"   ⚠️ Archivo existente corrupto: {mensaje}")
            print(f"   🗑️ Eliminando archivo corrupto...")
            os.remove(filename)
    
    # Configurar cliente con reintentos
    c = cdsapi.Client()
    
    # Preparar parámetros de la solicitud
    request_params = {
        'variable': variables,
        'year': year,
        'month': month,
        'day': day,
        'time': TIMES,
        'format': 'netcdf',
        'area': area
    }
    
    # Para ERA5 single-levels se necesita product_type
    if dataset == 'reanalysis-era5-single-levels':
        request_params['product_type'] = 'reanalysis'
    
    # Intentar descarga
    for intento in range(1, max_intentos + 1):
        print(f"\n   🔄 Intento {intento}/{max_intentos}")
        
        try:
            # Realizar descarga
            c.retrieve(dataset, request_params, filename)
            
            # Esperar un momento para que el archivo se escriba
            time.sleep(2)
            
            # Verificar el archivo descargado
            es_valido, mensaje = verificar_archivo_nc(filename)
            
            if es_valido:
                print(f"   ✅ Descarga exitosa: {mensaje}")
                return True
            else:
                print(f"   ❌ Verificación falló: {mensaje}")
                
                # Si el archivo existe pero está corrupto, eliminarlo
                if os.path.exists(filename):
                    os.remove(filename)
                
                # Si no es el último intento, esperar antes de reintentar
                if intento < max_intentos:
                    tiempo_espera = 5 * intento  # Espera progresiva
                    print(f"   ⏳ Esperando {tiempo_espera} segundos antes de reintentar...")
                    time.sleep(tiempo_espera)
                    
        except Exception as e:
            print(f"   ❌ Error en descarga: {str(e)}")
            
            # Limpiar archivo corrupto si existe
            if os.path.exists(filename):
                os.remove(filename)
            
            # Si no es el último intento, esperar
            if intento < max_intentos:
                tiempo_espera = 5 * intento
                print(f"   ⏳ Esperando {tiempo_espera} segundos antes de reintentar...")
                time.sleep(tiempo_espera)
    
    print(f"   ❌ No se pudo descargar después de {max_intentos} intentos")
    return False

def limpiar_archivos_corruptos():
    """Elimina archivos corruptos del directorio de descarga"""
    print(f"\n🧹 Limpiando archivos corruptos...")
    
    archivos_nc = glob.glob(os.path.join(DOWNLOAD_DIR, "*.nc"))
    eliminados = 0
    
    for archivo in archivos_nc:
        es_valido, mensaje = verificar_archivo_nc(archivo)
        if not es_valido:
            print(f"   🗑️ Eliminando: {os.path.basename(archivo)} - {mensaje}")
            os.remove(archivo)
            eliminados += 1
    
    if eliminados == 0:
        print(f"   ✅ No se encontraron archivos corruptos")
    else:
        print(f"   ✅ Eliminados {eliminados} archivos corruptos")
    
    return eliminados

# ==============================================================================
# DESCARGA PRINCIPAL
# ==============================================================================

def descargar_todos():
    """Descarga todos los grupos de variables"""
    
    print("\n" + "="*70)
    print("🚀 INICIANDO DESCARGA DE DATOS ERA5")
    print("="*70)
    print(f"\n📅 Fecha: {YEAR}-{MONTH}-{DAY}")
    print(f"📍 Punto tierra: {LAT_LAND}, {LON_LAND}")
    print(f"📍 Punto océano: {LAT_OCEAN}, {LON_OCEAN}")
    print(f"📂 Directorio: {DOWNLOAD_DIR}")
    
    # Primero, limpiar archivos corruptos existentes
    limpiar_archivos_corruptos()
    
    # Definir archivos a descargar
    archivos = [
        {
            'dataset': 'reanalysis-era5-land',
            'variables': VARS_GROUP_A,
            'area': AREA_LAND,
            'filename': os.path.join(DOWNLOAD_DIR, f"era5land_groupA_{YEAR}{MONTH}{DAY}.nc"),
            'nombre': 'Grupo A (Temperatura, Viento, Suelo)'
        },
        {
            'dataset': 'reanalysis-era5-land',
            'variables': VARS_GROUP_B,
            'area': AREA_LAND,
            'filename': os.path.join(DOWNLOAD_DIR, f"era5land_groupB_{YEAR}{MONTH}{DAY}.nc"),
            'nombre': 'Grupo B (Precipitación, Radiación)'
        },
        {
            'dataset': 'reanalysis-era5-single-levels',
            'variables': VARS_GROUP_C,
            'area': AREA_OCEAN,
            'filename': os.path.join(DOWNLOAD_DIR, f"era5_single_levels_groupC_{YEAR}{MONTH}{DAY}.nc"),
            'nombre': 'Grupo C (Oleaje)'
        }
    ]
    
    resultados = []
    
    # Descargar cada archivo
    for archivo in archivos:
        print(f"\n\n📦 {archivo['nombre']}")
        print("-" * 50)
        
        exito = descargar_con_verificacion(
            dataset=archivo['dataset'],
            variables=archivo['variables'],
            year=YEAR,
            month=MONTH,
            day=DAY,
            area=archivo['area'],
            filename=archivo['filename']
        )
        
        resultados.append({
            'nombre': archivo['nombre'],
            'archivo': os.path.basename(archivo['filename']),
            'exito': exito
        })
        
        # Pequeña pausa entre descargas para no sobrecargar la API
        if archivo != archivos[-1]:
            print(f"\n   ⏳ Pausa de 3 segundos antes de siguiente descarga...")
            time.sleep(3)
    
    # Mostrar resumen final
    print("\n" + "="*70)
    print("📊 RESUMEN DE DESCARGAS")
    print("="*70)
    
    todos_exitosos = True
    for resultado in resultados:
        estado = "✅ ÉXITO" if resultado['exito'] else "❌ FALLÓ"
        print(f"\n{estado} - {resultado['nombre']}")
        print(f"   Archivo: {resultado['archivo']}")
        if not resultado['exito']:
            todos_exitosos = False
    
    if todos_exitosos:
        print("\n" + "="*70)
        print("🎉 ¡TODAS LAS DESCARGAS COMPLETADAS EXITOSAMENTE!")
        print("="*70)
        print(f"\n📁 Los archivos están en: {DOWNLOAD_DIR}")
        
        # Verificar archivos finales
        print("\n🔍 Verificación final:")
        for archivo in archivos:
            if os.path.exists(archivo['filename']):
                es_valido, mensaje = verificar_archivo_nc(archivo['filename'])
                print(f"   {os.path.basename(archivo['filename'])}: {mensaje}")
    else:
        print("\n" + "="*70)
        print("⚠️ ALGUNAS DESCARGAS FALLARON")
        print("="*70)
        print("\n📝 Recomendaciones:")
        print("   1. Verifica tu conexión a internet")
        print("   2. Asegúrate de haber aceptado las licencias en la web de Copernicus")
        print("   3. Intenta ejecutar el script nuevamente")
        print("   4. Si el problema persiste, prueba descargar los archivos manualmente")

# ==============================================================================
# FUNCIÓN PARA VERIFICAR DESCARGA COMPLETA
# ==============================================================================

def verificar_descarga_completa():
    """Verifica que todos los archivos necesarios estén descargados correctamente"""
    
    print("\n" + "="*70)
    print("🔍 VERIFICANDO DESCARGA COMPLETA")
    print("="*70)
    
    archivos_esperados = [
        f"era5land_groupA_{YEAR}{MONTH}{DAY}.nc",
        f"era5land_groupB_{YEAR}{MONTH}{DAY}.nc",
        f"era5_single_levels_groupC_{YEAR}{MONTH}{DAY}.nc"
    ]
    
    todos_validos = True
    
    for archivo in archivos_esperados:
        ruta_completa = os.path.join(DOWNLOAD_DIR, archivo)
        
        if os.path.exists(ruta_completa):
            es_valido, mensaje = verificar_archivo_nc(ruta_completa)
            if es_valido:
                print(f"✅ {archivo}: {mensaje}")
            else:
                print(f"❌ {archivo}: {mensaje}")
                todos_validos = False
        else:
            print(f"❌ {archivo}: No encontrado")
            todos_validos = False
    
    return todos_validos

# ==============================================================================
# EJECUCIÓN
# ==============================================================================

if __name__ == "__main__":
    # Ejecutar descarga
    descargar_todos()
    
    # Verificar resultado
    print("\n")
    verificar_descarga_completa()
    
    print("\n" + "="*70)
    print("🏁 PROCESO COMPLETADO")
    print("="*70)


🚀 INICIANDO DESCARGA DE DATOS ERA5

📅 Fecha: 2024-09-20
📍 Punto tierra: 21.0, -86.9
📍 Punto océano: 21.0, -86.75
📂 Directorio: /Users/antonio/Downloads/data_era5_timeseries

🧹 Limpiando archivos corruptos...
   ✅ No se encontraron archivos corruptos


📦 Grupo A (Temperatura, Viento, Suelo)
--------------------------------------------------

📥 Descargando: era5land_groupA_20240920.nc

   🔄 Intento 1/3


2026-03-05 15:40:20,708 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land-timeseries?tab=overview)
2026-03-05 15:40:20,710 INFO Request ID is f551781f-a18e-4e92-96bc-6b0e8816fae1
2026-03-05 15:40:20,892 INFO status has been updated to accepted
2026-03-05 15:40:35,202 INFO status has been updated to successful


1593f13e0925aa3e5e03f25be57921ef.zip:   0%|          | 0.00/80.4k [00:00<?, ?B/s]

   ❌ Verificación falló: Archivo ZIP corrupto (80.44 KB)
   ⏳ Esperando 5 segundos antes de reintentar...

   🔄 Intento 2/3


2026-03-05 15:40:44,283 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land-timeseries?tab=overview)
2026-03-05 15:40:44,284 INFO Request ID is 24777825-9c91-4910-b3dc-14f3ca5d3f3d
2026-03-05 15:40:44,485 INFO status has been updated to accepted
2026-03-05 15:40:53,604 INFO status has been updated to running
2026-03-05 15:40:58,870 INFO status has been updated to successful


...

77e6b022e361619614ccf898f2407976.zip:   0%|          | 0.00/33.9k [00:00<?, ?B/s]

   ❌ Verificación falló: Archivo ZIP corrupto (33.94 KB)
   ⏳ Esperando 10 segundos antes de reintentar...

   🔄 Intento 3/3


2026-03-05 15:42:26,808 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land-timeseries?tab=overview)
2026-03-05 15:42:26,810 INFO Request ID is dae3813e-27b4-4375-b4cb-23bfa8290222
2026-03-05 15:42:26,992 INFO status has been updated to accepted
2026-03-05 15:42:41,049 INFO status has been updated to running
2026-03-05 15:42:48,823 INFO status has been updated to successful


77e6b022e361619614ccf898f2407976.zip:   0%|          | 0.00/33.9k [00:00<?, ?B/s]

   ❌ Verificación falló: Archivo ZIP corrupto (33.94 KB)
   ❌ No se pudo descargar después de 3 intentos

   ⏳ Pausa de 3 segundos antes de siguiente descarga...


📦 Grupo C (Oleaje)
--------------------------------------------------

📥 Descargando: era5_single_levels_groupC_20240920.nc
   ✅ Archivo existente válido: NetCDF4 válido (52.81 KB)

📊 RESUMEN DE DESCARGAS

❌ FALLÓ - Grupo A (Temperatura, Viento, Suelo)
   Archivo: era5land_groupA_20240920.nc

❌ FALLÓ - Grupo B (Precipitación, Radiación)
   Archivo: era5land_groupB_20240920.nc

✅ ÉXITO - Grupo C (Oleaje)
   Archivo: era5_single_levels_groupC_20240920.nc

⚠️ ALGUNAS DESCARGAS FALLARON

📝 Recomendaciones:
   1. Verifica tu conexión a internet
   2. Asegúrate de haber aceptado las licencias en la web de Copernicus
   3. Intenta ejecutar el script nuevamente
   4. Si el problema persiste, prueba descargar los archivos manualmente



🔍 VERIFICANDO DESCARGA COMPLETA
❌ era5land_groupA_20240920.nc: No encontrado
❌ era5land_groupB_202